In [1]:
from dbrepo.RestClient import RestClient
from dbrepo.api.dto import (
    QueryDefinition,
    FilterDefinition,
    FilterType,
    OrderDefinition,
    OrderType,
)
import pandas as pd
import time
import requests

In [ ]:
ENDPOINT = "https://test.dbrepo.tuwien.ac.at" 
USERNAME = "12549571"                          
PASSWORD = "********"                         
DB_ID = "3d81c073-e5fd-49b9-9536-b75ed490ca3e" 
API_BASE  = 'https://test.dbrepo.tuwien.ac.at/api/v1'                    

client = RestClient(endpoint=ENDPOINT, username=USERNAME, password=PASSWORD)


In [3]:
tables_url = f"{API_BASE}/database/{DB_ID}/table"
tables_response = requests.get(tables_url, auth=(USERNAME, PASSWORD), verify=True)

if tables_response.status_code == 200:
    tables_data = tables_response.json()
    print("Connection successful. Fetching table attributes...\n" + "-" * 40)
    
    for table in tables_data:
        table_id = table.get('id')
        table_name = table.get('name')
        
        if not table_id:
            continue
            
        # Use the table details endpoint to retrieve column metadata
        table_detail_url = f"{API_BASE}/database/{DB_ID}/table/{table_id}"
        table_response = requests.get(table_detail_url, auth=(USERNAME, PASSWORD), verify=True)
        
        if table_response.status_code == 200:
            full_table = table_response.json()
            print(f"\nTable: {table_name}")
            
            # Extract columns from the table details
            for col in full_table.get('columns', []):
                print(f"   - {col.get('name')} (Type: {col.get('type')})")
        else:
            print(f"Failed to read {table_name}. Status code: {table_response.status_code}")
                
    print("\n" + "-" * 40 + "\nExtraction complete!")
else:
    print(f"Connection error: status code {tables_response.status_code}")

Connection successful. Fetching table attributes...
----------------------------------------

Table: vehicle
   - vehicle_id (Type: double)
   - collision_index (Type: varchar)
   - vehicle_reference (Type: double)
   - vehicle_type (Type: varchar)
   - towing_and_articulation (Type: varchar)
   - vehicle_manoeuvre (Type: varchar)
   - vehicle_direction_from (Type: varchar)
   - vehicle_direction_to (Type: varchar)
   - vehicle_location_restricted_lane (Type: varchar)
   - junction_location (Type: varchar)
   - skidding_and_overturning (Type: varchar)
   - hit_object_in_carriageway (Type: varchar)
   - vehicle_leaving_carriageway (Type: varchar)
   - hit_object_off_carriageway (Type: varchar)
   - first_point_of_impact (Type: varchar)
   - vehicle_left_hand_drive (Type: double)
   - journey_purpose_of_driver (Type: varchar)
   - sex_of_driver (Type: varchar)
   - age_of_driver (Type: double)
   - age_band_of_driver (Type: varchar)
   - engine_capacity_cc (Type: double)
   - propulsion_

In [4]:
import requests

# Master dictionary containing all semantic links used across the project
semantic_mappings = {
    # --- INFRASTRUCTURE AND TIME (OSM / OWL-Time) ---
    "road_type": "https://wiki.openstreetmap.org/wiki/Key:highway",
    "first_road_class": "https://wiki.openstreetmap.org/wiki/Key:highway",
    "second_road_class": "https://wiki.openstreetmap.org/wiki/Key:highway",
    "trunk_road_flag": "https://wiki.openstreetmap.org/wiki/Key:highway",
    "first_road_number": "https://wiki.openstreetmap.org/wiki/Key:ref",
    "second_road_number": "https://wiki.openstreetmap.org/wiki/Key:ref",
    "speed_limit": "https://wiki.openstreetmap.org/wiki/Key:maxspeed",
    "pedestrian_crossing": "https://wiki.openstreetmap.org/wiki/Key:crossing",
    "road_surface_conditions": "https://wiki.openstreetmap.org/wiki/Key:surface",
    "collision_year": "http://www.w3.org/2006/time#year",
    "date": "http://www.w3.org/2006/time#Instant",
    "time": "http://www.w3.org/2006/time#hour",
    "day_of_week": "http://www.w3.org/2006/time#DayOfWeek",

    # --- ENVIRONMENT (SOSA) ---
    "weather_conditions": "http://www.w3.org/ns/sosa/ObservableProperty",
    "light_conditions": "http://www.w3.org/ns/sosa/ObservableProperty",
    "special_conditions_at_site": "http://www.w3.org/ns/sosa/ObservableProperty",

    # --- DEMOGRAPHICS (FOAF) ---
    "sex_of_driver": "http://xmlns.com/foaf/0.1/gender",
    "age_of_driver": "http://xmlns.com/foaf/0.1/age",
    "age_band_of_driver": "http://xmlns.com/foaf/0.1/age",
    "sex_of_casualty": "http://xmlns.com/foaf/0.1/gender",
    "age_of_casualty": "http://xmlns.com/foaf/0.1/age",
    "age_band_of_casualty": "http://xmlns.com/foaf/0.1/age",

    # --- VEHICLES AND MECHANICS (VSSo) ---
    "vehicle_type": "http://automotive.eurecom.fr/vsso#Vehicle",
    "engine_capacity_cc": "http://automotive.eurecom.fr/vsso#EngineDisplacement",
    "propulsion_code": "http://automotive.eurecom.fr/vsso#Powertrain",
    "vehicle_left_hand_drive": "http://automotive.eurecom.fr/vsso#SteeringWheelPosition",
    "generic_make_model": "http://automotive.eurecom.fr/vsso#Model",
    "age_of_vehicle": "http://automotive.eurecom.fr/vsso#Vehicle",
    "escooter_flag": "http://automotive.eurecom.fr/vsso#Vehicle",

    # --- ACCIDENTS AND TRAFFIC (DATEX II) ---
    "vehicle_manoeuvre": "http://cef.uv.es/lodroadtran18/def/transporte/dtx_srti#VehicleManoeuvreEnum",
    "vehicle_direction_from": "http://cef.uv.es/lodroadtran18/def/transporte/dtx_srti#direction",
    "vehicle_direction_to": "http://cef.uv.es/lodroadtran18/def/transporte/dtx_srti#direction",
    "vehicle_location_restricted_lane": "http://cef.uv.es/lodroadtran18/def/transporte/dtx_srti#LaneEnum",
    "junction_location": "http://cef.uv.es/lodroadtran18/def/transporte/dtx_srti#Junction",
    "junction_detail": "http://cef.uv.es/lodroadtran18/def/transporte/dtx_srti#Junction",
    "junction_control": "http://cef.uv.es/lodroadtran18/def/transporte/dtx_srti#Junction",
    "skidding_and_overturning": "http://cef.uv.es/lodroadtran18/def/transporte/dtx_srti#Accident",
    "hit_object_in_carriageway": "http://cef.uv.es/lodroadtran18/def/transporte/dtx_srti#Accident",
    "vehicle_leaving_carriageway": "http://cef.uv.es/lodroadtran18/def/transporte/dtx_srti#Accident",
    "hit_object_off_carriageway": "http://cef.uv.es/lodroadtran18/def/transporte/dtx_srti#Accident",
    "first_point_of_impact": "http://cef.uv.es/lodroadtran18/def/transporte/dtx_srti#Accident",
    "towing_and_articulation": "http://cef.uv.es/lodroadtran18/def/transporte/dtx_srti#VehicleEquipmentEnum",
    "collision_severity": "http://cef.uv.es/lodroadtran18/def/transporte/dtx_srti#Accident",
    "enhanced_severity_collision": "http://cef.uv.es/lodroadtran18/def/transporte/dtx_srti#Accident",
    "collision_injury_based": "http://cef.uv.es/lodroadtran18/def/transporte/dtx_srti#Accident",
    "collision_adjusted_severity_serious": "http://cef.uv.es/lodroadtran18/def/transporte/dtx_srti#Accident",
    "collision_adjusted_severity_slight": "http://cef.uv.es/lodroadtran18/def/transporte/dtx_srti#Accident",
    "number_of_vehicles": "http://cef.uv.es/lodroadtran18/def/transporte/dtx_srti#numberOfVehiclesInvolved",
    "number_of_casualties": "http://cef.uv.es/lodroadtran18/def/transporte/dtx_srti#Accident",
    "carriageway_hazards": "http://cef.uv.es/lodroadtran18/def/transporte/dtx_srti#Obstruction",
    "casualty_class": "http://cef.uv.es/lodroadtran18/def/transporte/dtx_srti#InjuryStatusEnum",
    "casualty_severity": "http://cef.uv.es/lodroadtran18/def/transporte/dtx_srti#InjuryStatusEnum",
    "enhanced_casualty_severity": "http://cef.uv.es/lodroadtran18/def/transporte/dtx_srti#InjuryStatusEnum",
    "casualty_adjusted_severity_serious": "http://cef.uv.es/lodroadtran18/def/transporte/dtx_srti#InjuryStatusEnum",
    "casualty_adjusted_severity_slight": "http://cef.uv.es/lodroadtran18/def/transporte/dtx_srti#InjuryStatusEnum",
    "casualty_injury_based": "http://cef.uv.es/lodroadtran18/def/transporte/dtx_srti#Accident",
    "casualty_type": "http://cef.uv.es/lodroadtran18/def/transporte/dtx_srti#Person",
    "car_passenger": "http://cef.uv.es/lodroadtran18/def/transporte/dtx_srti#Occupant",
    "bus_or_coach_passenger": "http://cef.uv.es/lodroadtran18/def/transporte/dtx_srti#Occupant",
    "pedestrian_location": "http://cef.uv.es/lodroadtran18/def/transporte/dtx_srti#Person",
    "pedestrian_movement": "http://cef.uv.es/lodroadtran18/def/transporte/dtx_srti#Person",
    "pedestrian_road_maintenance_worker": "http://cef.uv.es/lodroadtran18/def/transporte/dtx_srti#Person",

    # --- IDENTIFIERS, ADMINISTRATION, AND MISCELLANEOUS (Schema.org / W3C) ---
    "vehicle_id": "https://schema.org/identifier",
    "collision_index": "https://schema.org/identifier",
    "vehicle_reference": "https://schema.org/identifier",
    "casualty_id": "https://schema.org/identifier",
    "casualty_reference": "https://schema.org/identifier",
    "collision_ref_no": "https://schema.org/identifier",
    "journey_purpose_of_driver": "https://schema.org/Trip",
    "driver_imd_decile": "https://schema.org/Rating",
    "lsoa_of_driver": "https://schema.org/Place",
    "driver_distance_banding": "https://schema.org/Distance",
    "longitude": "http://www.w3.org/2003/01/geo/wgs84_pos#long",
    "latitude": "http://www.w3.org/2003/01/geo/wgs84_pos#lat",
    "location_easting_osgr": "https://schema.org/geo",
    "location_northing_osgr": "https://schema.org/geo",
    "urban_or_rural_area": "https://schema.org/Place",
    "police_force": "https://schema.org/Organization",
    "did_police_officer_attend_scene_of_accident": "https://schema.org/Observation",
    "local_authority_district": "https://schema.org/AdministrativeArea",
    "local_authority_ons_district": "https://schema.org/AdministrativeArea",
    "local_authority_highway": "https://schema.org/AdministrativeArea",
    "local_authority_highway_current": "https://schema.org/AdministrativeArea",
    "lsoa_of_accident_location": "https://schema.org/AdministrativeArea",
    "casualty_imd_decile": "https://schema.org/Rating",
    "lsoa_of_casualty": "https://schema.org/Place",
    "casualty_distance_banding": "https://schema.org/Distance"
}

# Justification of ontology choices

## 1. Road Infrastructure (OSM Ontology)
"The OpenStreetMap (OSM) ontology was chosen to describe the physical infrastructure (road types, speed limits, surface conditions). OSM is a prominent and robust global standard for spatial data, providing precise semantics for cartographic elements without relying on generic ontologies."

## 2. Environmental Data (SOSA Ontology)
"Attributes related to weather and lighting were mapped to the ObservableProperty concept of the W3C SOSA ontology. These variables describe fluctuating environmental conditions captured at the time of the crash; SOSA is the ideal standard for formally representing this type of observation."

## 3. Temporality (OWL-Time Ontology)
"For dates, times, and days of the week, the OWL-Time ontology was selected. It is the undisputed W3C standard for representing temporal concepts, ensuring universal interoperability for the experiment's timestamps."

## 4. Vehicle Characteristics (VSSo Ontology)
"Mechanical and descriptive vehicle attributes (engine type, steering wheel position, age) are mapped using the Vehicle Signal Specification Ontology (VSSo). VSSo is a highly specialized ontology backed by the automotive industry, perfectly suited for strict domain granularity."

## 5. Collision Dynamics and Casualties (DATEX II Ontology)
"DATEX II was favored for modeling events (impacts, maneuvers) and human consequences (injury severity, passenger status). As the official European standard for intelligent transport systems and traffic management, it provides precise and rigorous domain semantics for road accident data."

## 6. Administrative Metadata and Identifiers (Schema.org)
"In the absence of domain-specific ontologies for simple primary keys (id) or administrative classifications (regional codes), the generic Schema.org vocabulary was used as a fallback solution. This choice ensures the semantic interoperability of these structural metadata."

In [5]:
# Build a semantic mapping report from the DBRepo API metadata
project_tables = {"collision", "vehicle", "casualty"}
mapping_rows = []

tables_url = f"{API_BASE}/database/{DB_ID}/table"
tables_response = requests.get(tables_url, auth=(USERNAME, PASSWORD), verify=True)

if tables_response.status_code != 200:
    raise RuntimeError(f"Failed to retrieve tables list: {tables_response.status_code} - {tables_response.text}")

for table in tables_response.json():
    table_id = table.get("id")
    table_name = table.get("name", "unknown_table")

    if not table_id or table_name not in project_tables:
        continue

    table_detail_url = f"{API_BASE}/database/{DB_ID}/table/{table_id}"
    table_response = requests.get(table_detail_url, auth=(USERNAME, PASSWORD), verify=True)
    if table_response.status_code != 200:
        print(f"Warning: unable to read metadata for {table_name} ({table_response.status_code})")
        continue

    table_payload = table_response.json()
    for column in table_payload.get("columns", []):
        column_name = column.get("name")
        semantic_uri = semantic_mappings.get(column_name)
        mapping_rows.append({
            "table_name": table_name,
            "column_name": column_name,
            "dbrepo_type": column.get("type"),
            "semantic_uri": semantic_uri,
            "mapped": semantic_uri is not None,
        })

semantic_mapping_report = pd.DataFrame(mapping_rows)

display(semantic_mapping_report)

print(f"Mapped columns: {semantic_mapping_report['mapped'].sum()} / {len(semantic_mapping_report)}")
if (~semantic_mapping_report["mapped"]).any():
    print("Unmapped columns:")
    for column_name in semantic_mapping_report.loc[~semantic_mapping_report["mapped"], "column_name"].dropna().unique():
        print(f" - {column_name}")

,table_name,column_name,dbrepo_type,semantic_uri,mapped
0,vehicle,vehicle_id,double,https://schema.org/identifier,True
1,vehicle,collision_index,varchar,https://schema.org/identifier,True
2,vehicle,vehicle_reference,double,https://schema.org/identifier,True
3,vehicle,vehicle_type,varchar,http://automotive.eurecom.fr/vsso#Vehicle,True
4,vehicle,towing_and_articulation,varchar,http://cef.uv.es/lodroadtran18/def/transporte/...,True
...,...,...,...,...,...
85,casualty,enhanced_casualty_severity,varchar,http://cef.uv.es/lodroadtran18/def/transporte/...,True
86,casualty,casualty_injury_based,double,http://cef.uv.es/lodroadtran18/def/transporte/...,True
87,casualty,casualty_adjusted_severity_serious,double,http://cef.uv.es/lodroadtran18/def/transporte/...,True
88,casualty,casualty_adjusted_severity_slight,double,http://cef.uv.es/lodroadtran18/def/transporte/...,True


Mapped columns: 90 / 90


In [6]:
# Write semantic concept links back to DBRepo
apply_semantic_links = True
update_rows = []

tables_url = f"{API_BASE}/database/{DB_ID}/table"
tables_response = requests.get(tables_url, auth=(USERNAME, PASSWORD), verify=True)

if tables_response.status_code != 200:
    raise RuntimeError(f"Failed to retrieve tables list: {tables_response.status_code} - {tables_response.text}")

for table in tables_response.json():
    table_id = table.get("id")
    table_name = table.get("name", "unknown_table")

    if not table_id or table_name not in project_tables:
        continue

    table_detail_url = f"{API_BASE}/database/{DB_ID}/table/{table_id}"
    table_response = requests.get(table_detail_url, auth=(USERNAME, PASSWORD), verify=True)
    if table_response.status_code != 200:
        print(f"Warning: unable to read metadata for {table_name} ({table_response.status_code})")
        continue

    table_payload = table_response.json()
    for column in table_payload.get("columns", []):
        column_id = column.get("id")
        column_name = column.get("name")
        concept_uri = semantic_mappings.get(column_name)

        if not column_id:
            update_rows.append({
                "table_name": table_name,
                "column_name": column_name,
                "concept_uri": concept_uri,
                "status": "missing_column_id",
            })
            continue

        if not concept_uri:
            update_rows.append({
                "table_name": table_name,
                "column_name": column_name,
                "concept_uri": None,
                "status": "no_semantic_mapping",
            })
            continue

        if apply_semantic_links:
            try:
                updated_column = client.update_table_column(
                    database_id=DB_ID,
                    table_id=table_id,
                    column_id=column_id,
                    concept_uri=concept_uri,
                )
                update_rows.append({
                    "table_name": table_name,
                    "column_name": column_name,
                    "concept_uri": concept_uri,
                    "status": "updated",
                    "returned_concept_uri": getattr(updated_column, "concept_uri", None),
                })
                print(f"Updated {table_name}.{column_name} -> {concept_uri}")
            except Exception as exc:
                update_rows.append({
                    "table_name": table_name,
                    "column_name": column_name,
                    "concept_uri": concept_uri,
                    "status": f"failed: {exc}",
                })
                print(f"Failed to update {table_name}.{column_name}: {exc}")
        else:
            update_rows.append({
                "table_name": table_name,
                "column_name": column_name,
                "concept_uri": concept_uri,
                "status": "dry_run",
            })

semantic_link_update_report = pd.DataFrame(update_rows)
display(semantic_link_update_report)

print(f"Updated columns: {(semantic_link_update_report['status'] == 'updated').sum()}")
print(f"Skipped columns: {(semantic_link_update_report['status'] == 'no_semantic_mapping').sum()}")

Updated vehicle.vehicle_id -> https://schema.org/identifier
Updated vehicle.collision_index -> https://schema.org/identifier
Updated vehicle.vehicle_reference -> https://schema.org/identifier
Updated vehicle.vehicle_type -> http://automotive.eurecom.fr/vsso#Vehicle
Updated vehicle.towing_and_articulation -> http://cef.uv.es/lodroadtran18/def/transporte/dtx_srti#VehicleEquipmentEnum
Updated vehicle.vehicle_manoeuvre -> http://cef.uv.es/lodroadtran18/def/transporte/dtx_srti#VehicleManoeuvreEnum
Updated vehicle.vehicle_direction_from -> http://cef.uv.es/lodroadtran18/def/transporte/dtx_srti#direction
Updated vehicle.vehicle_direction_to -> http://cef.uv.es/lodroadtran18/def/transporte/dtx_srti#direction
Updated vehicle.vehicle_location_restricted_lane -> http://cef.uv.es/lodroadtran18/def/transporte/dtx_srti#LaneEnum
Updated vehicle.junction_location -> http://cef.uv.es/lodroadtran18/def/transporte/dtx_srti#Junction
Updated vehicle.skidding_and_overturning -> http://cef.uv.es/lodroadtran1

,table_name,column_name,concept_uri,status,returned_concept_uri
0,vehicle,vehicle_id,https://schema.org/identifier,updated,https://schema.org/identifier
1,vehicle,collision_index,https://schema.org/identifier,updated,https://schema.org/identifier
2,vehicle,vehicle_reference,https://schema.org/identifier,updated,https://schema.org/identifier
3,vehicle,vehicle_type,http://automotive.eurecom.fr/vsso#Vehicle,updated,http://automotive.eurecom.fr/vsso#Vehicle
4,vehicle,towing_and_articulation,http://cef.uv.es/lodroadtran18/def/transporte/...,updated,http://cef.uv.es/lodroadtran18/def/transporte/...
...,...,...,...,...,...
85,casualty,enhanced_casualty_severity,http://cef.uv.es/lodroadtran18/def/transporte/...,updated,http://cef.uv.es/lodroadtran18/def/transporte/...
86,casualty,casualty_injury_based,http://cef.uv.es/lodroadtran18/def/transporte/...,updated,http://cef.uv.es/lodroadtran18/def/transporte/...
87,casualty,casualty_adjusted_severity_serious,http://cef.uv.es/lodroadtran18/def/transporte/...,updated,http://cef.uv.es/lodroadtran18/def/transporte/...
88,casualty,casualty_adjusted_severity_slight,http://cef.uv.es/lodroadtran18/def/transporte/...,updated,http://cef.uv.es/lodroadtran18/def/transporte/...


Updated columns: 90
Skipped columns: 0


In [7]:
# Verify that semantic concept links were written successfully
verification_rows = []

for table in client.get_tables(database_id=DB_ID):
    table_name = table.name
    if table_name not in project_tables:
        continue

    table_payload = client.get_table(database_id=DB_ID, table_id=table.id)
    for column in table_payload.columns:
        column_name = column.name
        expected_concept_uri = semantic_mappings.get(column_name)
        actual_concept_uri = getattr(column, "concept_uri", None)
        verification_rows.append({
            "table_name": table_name,
            "column_name": column_name,
            "expected_concept_uri": expected_concept_uri,
            "actual_concept_uri": actual_concept_uri,
            "matches": expected_concept_uri == actual_concept_uri,
        })

semantic_link_verification_report = pd.DataFrame(verification_rows)
display(semantic_link_verification_report)

print(f"Matching links: {semantic_link_verification_report['matches'].sum()} / {len(semantic_link_verification_report)}")
if (~semantic_link_verification_report["matches"]).any():
    print("Mismatched or missing links:")
    for _, row in semantic_link_verification_report.loc[~semantic_link_verification_report["matches"]].iterrows():
        print(f" - {row['table_name']}.{row['column_name']}")

,table_name,column_name,expected_concept_uri,actual_concept_uri,matches
0,vehicle,vehicle_id,https://schema.org/identifier,https://schema.org/identifier,True
1,vehicle,collision_index,https://schema.org/identifier,https://schema.org/identifier,True
2,vehicle,vehicle_reference,https://schema.org/identifier,https://schema.org/identifier,True
3,vehicle,vehicle_type,http://automotive.eurecom.fr/vsso#Vehicle,http://automotive.eurecom.fr/vsso#Vehicle,True
4,vehicle,towing_and_articulation,http://cef.uv.es/lodroadtran18/def/transporte/...,http://cef.uv.es/lodroadtran18/def/transporte/...,True
...,...,...,...,...,...
85,casualty,enhanced_casualty_severity,http://cef.uv.es/lodroadtran18/def/transporte/...,http://cef.uv.es/lodroadtran18/def/transporte/...,True
86,casualty,casualty_injury_based,http://cef.uv.es/lodroadtran18/def/transporte/...,http://cef.uv.es/lodroadtran18/def/transporte/...,True
87,casualty,casualty_adjusted_severity_serious,http://cef.uv.es/lodroadtran18/def/transporte/...,http://cef.uv.es/lodroadtran18/def/transporte/...,True
88,casualty,casualty_adjusted_severity_slight,http://cef.uv.es/lodroadtran18/def/transporte/...,http://cef.uv.es/lodroadtran18/def/transporte/...,True


Matching links: 90 / 90
